# Neo4j Simple Connection Test

Quick validation of Neo4j connectivity: network (TCP) and Python driver.

---

## Configuration

Set your Neo4j Aura credentials below.

In [ ]:
# =============================================================================
# CONFIGURATION - Edit these values for your Neo4j instance
# =============================================================================

NEO4J_URI = "neo4j+s://123456.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""
NEO4J_DATABASE = "neo4j"

# Path to the connector JAR in a UC Volume
JDBC_JAR_PATH = "/Volumes/<catalog>/<schema>/<volume>/neo4j-unity-catalog-connector.jar"

# Name for the Unity Catalog connection
UC_CONNECTION_NAME = "<catalog>.<schema>.neo4j_jdbc"

# Derived values (no need to edit)
NEO4J_HOST = NEO4J_URI.replace("neo4j+s://", "")
JAVA_DEPENDENCIES = f'["{JDBC_JAR_PATH}"]'
NEO4J_JDBC_URL = f"jdbc:neo4j+s://{NEO4J_HOST}:7687/{NEO4J_DATABASE}"
NEO4J_JDBC_URL_SQL = f"{NEO4J_JDBC_URL}?enableSQLTranslation=true"

print("Configuration:")
print(f"  Neo4j URI: {NEO4J_URI}")
print(f"  Neo4j Host: {NEO4J_HOST}")
print(f"  Username: {NEO4J_USERNAME}")
print(f"  JDBC URL: {NEO4J_JDBC_URL_SQL}")
print(f"  UC Connection: {UC_CONNECTION_NAME}")
print(f"  JDBC JAR Path: {JDBC_JAR_PATH}")

---

## Section 1: Network Connectivity Test (TCP Layer)

**Expected Result**: PASS - Proves network path is open.

In [ ]:
# TCP connectivity test using netcat
import subprocess
import time

print("=" * 60)
print("TEST: Network Connectivity (TCP)")
print("=" * 60)
print(f"\nTarget: {NEO4J_HOST}:7687 (Bolt protocol port)")
print("Testing: Can we reach Neo4j at the network level?")

try:
    start_time = time.time()
    result = subprocess.run(
        ['nc', '-zv', NEO4J_HOST, '7687'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10
    )
    elapsed = (time.time() - start_time) * 1000
    output = (result.stdout.decode() + result.stderr.decode()).strip()

    if result.returncode == 0:
        print("\n" + "=" * 60)
        print(">>> CONNECTIVITY VERIFIED <<<")
        print("=" * 60)
        print(f"\n[PASS] TCP connection established in {elapsed:.1f}ms")
        print(f"\nConnection Details:")
        print(f"  - Host: {NEO4J_HOST}")
        print(f"  - Port: 7687 (Bolt)")
        print(f"  - TCP Latency: {elapsed:.1f}ms")
        if output:
            print(f"  - Raw Output: {output}")
        print("\n" + "-" * 60)
        print("RESULT: Network path to Neo4j is OPEN")
        print("        Firewall rules allow Bolt protocol traffic")
        print("-" * 60)
        print("\nStatus: PASS")
    else:
        print(f"\n[FAIL] Cannot reach {NEO4J_HOST}:7687 - check firewall rules")
        print(f"Details: {output}")
        print("\nStatus: FAIL")

except Exception as e:
    print(f"\n[FAIL] Error: {e}")
    print("\nStatus: FAIL")

---

## Section 2: Neo4j Python Driver Test

**Expected Result**: PASS - Proves credentials work and Neo4j is accessible.

In [ ]:
# Test Neo4j Python driver connectivity
import time

print("=" * 60)
print("TEST: Neo4j Python Driver")
print("=" * 60)
print(f"\nTarget: {NEO4J_URI}")
print("Testing: Can we authenticate and execute queries via Bolt protocol?")

from neo4j import GraphDatabase

try:
    start_time = time.time()
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
    
    # Verify connectivity
    driver.verify_connectivity()
    connect_time = (time.time() - start_time) * 1000
    
    print("\n" + "=" * 60)
    print(">>> AUTHENTICATION SUCCESSFUL <<<")
    print("=" * 60)
    print(f"\n[PASS] Driver connected and authenticated in {connect_time:.1f}ms")
    
    # Test simple query
    with driver.session() as session:
        query_start = time.time()
        result = session.run("RETURN 1 AS test")
        record = result.single()
        query_time = (time.time() - query_start) * 1000
        print(f"[PASS] Query executed: RETURN 1 = {record['test']} ({query_time:.1f}ms)")
        
        # Get Neo4j version
        result = session.run("CALL dbms.components() YIELD name, versions RETURN name, versions")
        neo4j_info = []
        for record in result:
            neo4j_info.append(f"{record['name']} {record['versions']}")
    
    total_time = (time.time() - start_time) * 1000
    driver.close()
    
    print(f"\nConnection Details:")
    print(f"  - URI: {NEO4J_URI}")
    print(f"  - User: {NEO4J_USERNAME}")
    print(f"  - Neo4j Server: {', '.join(neo4j_info)}")
    print(f"  - Connection Time: {connect_time:.1f}ms")
    print(f"  - Total Test Time: {total_time:.1f}ms")
    
    print("\n" + "-" * 60)
    print("RESULT: Neo4j Python Driver connection WORKING")
    print("        Credentials valid, Bolt protocol functional")
    print("-" * 60)
    print("\nStatus: PASS")
    
except Exception as e:
    print(f"\n[FAIL] Connection failed: {e}")
    print("\nStatus: FAIL")

---

## Section 3: Create Sample Test Data

Creates a small graph of people, movies, and relationships to validate write operations and provide data for query testing.

In [ ]:
# Create sample test data
from neo4j import GraphDatabase

print("=" * 60)
print("SETUP: Creating Sample Test Data")
print("=" * 60)

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

try:
    with driver.session() as session:
        # Clear any existing test data
        session.run("MATCH (n) WHERE n:Person OR n:Movie DETACH DELETE n")
        print("\n[OK] Cleared existing Person/Movie nodes")

        # Create Person nodes
        session.run("""
            CREATE (alice:Person {name: 'Alice', age: 34, role: 'Engineer'})
            CREATE (bob:Person {name: 'Bob', age: 28, role: 'Designer'})
            CREATE (carol:Person {name: 'Carol', age: 42, role: 'Manager'})
            CREATE (dave:Person {name: 'Dave', age: 31, role: 'Engineer'})
            CREATE (eve:Person {name: 'Eve', age: 26, role: 'Data Scientist'})
        """)
        print("[OK] Created 5 Person nodes")

        # Create Movie nodes
        session.run("""
            CREATE (m1:Movie {title: 'The Matrix', year: 1999, genre: 'Sci-Fi'})
            CREATE (m2:Movie {title: 'Inception', year: 2010, genre: 'Sci-Fi'})
            CREATE (m3:Movie {title: 'The Shawshank Redemption', year: 1994, genre: 'Drama'})
        """)
        print("[OK] Created 3 Movie nodes")

        # Create KNOWS relationships between people
        session.run("""
            MATCH (alice:Person {name: 'Alice'}), (bob:Person {name: 'Bob'})
            CREATE (alice)-[:KNOWS {since: 2020}]->(bob)
        """)
        session.run("""
            MATCH (alice:Person {name: 'Alice'}), (carol:Person {name: 'Carol'})
            CREATE (alice)-[:KNOWS {since: 2018}]->(carol)
        """)
        session.run("""
            MATCH (bob:Person {name: 'Bob'}), (dave:Person {name: 'Dave'})
            CREATE (bob)-[:KNOWS {since: 2021}]->(dave)
        """)
        session.run("""
            MATCH (carol:Person {name: 'Carol'}), (eve:Person {name: 'Eve'})
            CREATE (carol)-[:KNOWS {since: 2019}]->(eve)
        """)
        session.run("""
            MATCH (dave:Person {name: 'Dave'}), (eve:Person {name: 'Eve'})
            CREATE (dave)-[:KNOWS {since: 2022}]->(eve)
        """)
        print("[OK] Created 5 KNOWS relationships")

        # Create LIKES relationships to movies
        session.run("""
            MATCH (alice:Person {name: 'Alice'}), (m:Movie {title: 'The Matrix'})
            CREATE (alice)-[:LIKES {rating: 5}]->(m)
        """)
        session.run("""
            MATCH (alice:Person {name: 'Alice'}), (m:Movie {title: 'Inception'})
            CREATE (alice)-[:LIKES {rating: 4}]->(m)
        """)
        session.run("""
            MATCH (bob:Person {name: 'Bob'}), (m:Movie {title: 'Inception'})
            CREATE (bob)-[:LIKES {rating: 5}]->(m)
        """)
        session.run("""
            MATCH (carol:Person {name: 'Carol'}), (m:Movie {title: 'The Shawshank Redemption'})
            CREATE (carol)-[:LIKES {rating: 5}]->(m)
        """)
        session.run("""
            MATCH (dave:Person {name: 'Dave'}), (m:Movie {title: 'The Matrix'})
            CREATE (dave)-[:LIKES {rating: 4}]->(m)
        """)
        session.run("""
            MATCH (eve:Person {name: 'Eve'}), (m:Movie {title: 'Inception'})
            CREATE (eve)-[:LIKES {rating: 3}]->(m)
        """)
        print("[OK] Created 6 LIKES relationships")

        # Verify counts
        result = session.run("MATCH (n) RETURN labels(n)[0] AS label, count(*) AS count ORDER BY label")
        print(f"\nNode counts:")
        for record in result:
            print(f"  - {record['label']}: {record['count']}")

        result = session.run("MATCH ()-[r]->() RETURN type(r) AS type, count(*) AS count ORDER BY type")
        print(f"\nRelationship counts:")
        for record in result:
            print(f"  - {record['type']}: {record['count']}")

    print("\n" + "-" * 60)
    print("RESULT: Sample test data created successfully")
    print("        5 Person nodes, 3 Movie nodes")
    print("        5 KNOWS relationships, 6 LIKES relationships")
    print("-" * 60)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] Error creating test data: {e}")
    print("\nStatus: FAIL")
finally:
    driver.close()

---

## Section 4: Unity Catalog JDBC Connection

Create and test the UC JDBC connection. The SafeSpark sandbox wraps the JDBC driver in an isolated JVM.

**Note:** `java_dependencies` only accepts UC Volume paths.

In [ ]:
# Create Unity Catalog JDBC Connection
import time

print("=" * 60)
print("SETUP: Creating UC JDBC Connection")
print("=" * 60)
print(f"\nConnection Name: {UC_CONNECTION_NAME}")
print(f"JDBC URL: {NEO4J_JDBC_URL_SQL}")

spark.sql(f"DROP CONNECTION IF EXISTS {UC_CONNECTION_NAME}")

create_sql = f"""
CREATE CONNECTION {UC_CONNECTION_NAME} TYPE JDBC
ENVIRONMENT (
  java_dependencies '{JAVA_DEPENDENCIES}'
)
OPTIONS (
  url '{NEO4J_JDBC_URL_SQL}',
  user '{NEO4J_USERNAME}',
  password '{NEO4J_PASSWORD}',
  driver 'org.neo4j.jdbc.Neo4jDriver',
  externalOptionsAllowList 'dbtable,query,partitionColumn,lowerBound,upperBound,numPartitions,fetchSize,customSchema'
)
"""

try:
    start_time = time.time()
    spark.sql(create_sql)
    elapsed = (time.time() - start_time) * 1000

    print(f"\n[PASS] Connection created in {elapsed:.1f}ms")

    # Verify connection
    spark.sql(f"DESCRIBE CONNECTION {UC_CONNECTION_NAME}").show(truncate=False)

    # Test with SELECT 1
    print("Testing UC JDBC with SELECT 1...")
    start_time = time.time()
    df = spark.read.format("jdbc") \
        .option("databricks.connection", UC_CONNECTION_NAME) \
        .option("query", "SELECT 1 AS test") \
        .option("customSchema", "test INT") \
        .load()

    results = df.collect()
    elapsed = (time.time() - start_time) * 1000

    print(f"[PASS] UC JDBC query returned: {results[0]['test']} ({elapsed:.1f}ms)")

    print("\n" + "-" * 60)
    print("RESULT: UC JDBC connection created and verified")
    print("-" * 60)
    print("\nStatus: PASS")

except Exception as e:
    print(f"\n[FAIL] {e}")
    print("\nStatus: FAIL")

---

## Section 5: Query Sample Data via UC JDBC

Run SQL queries against the Neo4j test data through the Unity Catalog JDBC connection. SQL is automatically translated to Cypher by the connector.

**Note**: `customSchema` is required because Neo4j's JDBC driver returns `NullType` during Spark's schema inference.

In [ ]:
# Query sample data via UC JDBC
import time

print("=" * 60)
print("TEST: Query Sample Data via UC JDBC")
print("=" * 60)

def run_jdbc_query(description, sql, schema):
    """Helper to run a SQL query through the UC JDBC connection."""
    print(f"\n--- {description} ---")
    print(f"SQL: {sql}")
    try:
        start_time = time.time()
        df = spark.read.format("jdbc") \
            .option("databricks.connection", UC_CONNECTION_NAME) \
            .option("query", sql) \
            .option("customSchema", schema) \
            .load()
        elapsed = (time.time() - start_time) * 1000
        print(f"[PASS] Query returned in {elapsed:.1f}ms")
        df.show(truncate=False)
        return df
    except Exception as e:
        print(f"[FAIL] {e}")
        return None

# Query 1: Count all Person nodes
run_jdbc_query(
    "Count People",
    "SELECT COUNT(*) AS person_count FROM Person",
    "person_count LONG"
)

# Query 2: List all people
run_jdbc_query(
    "All People",
    "SELECT name, age, role FROM Person",
    "name STRING, age LONG, role STRING"
)

# Query 3: Count all Movie nodes
run_jdbc_query(
    "Count Movies",
    "SELECT COUNT(*) AS movie_count FROM Movie",
    "movie_count LONG"
)

# Query 4: All movies
run_jdbc_query(
    "All Movies",
    "SELECT title, year, genre FROM Movie",
    "title STRING, year LONG, genre STRING"
)

print("\n" + "-" * 60)
print("RESULT: UC JDBC queries executed successfully")
print("        SQL translated to Cypher by the connector")
print("-" * 60)
print("\nStatus: PASS")